In [ ]:
# Cell 1: Environment Setup, Dependencies & Repository Cloning
import os
import sys
import subprocess

print("[SETUP] Setting up Kaggle CPU Data Generator Environment (3,500 Shards Multi-Phase Dataset)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "sympy"], check=True)

repo_url = "https://github.com/dsainvg001/transformer-math.git"
repo_dir = "transformer-math"

if not os.path.exists(repo_dir):
    print(f"[GIT] Cloning repository from {repo_url}...")
    subprocess.run(["git", "clone", repo_url], check=True)
else:
    print(f"[GIT] Repository {repo_dir} already present.")

if os.path.exists(repo_dir):
    os.chdir(repo_dir)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f"[PATH] Current Directory: {os.getcwd()}")

In [ ]:
# Cell 2: Hugging Face Authentication Token Configuration
from getpass import getpass

hf_token = os.environ.get("HFTOKEN") or os.environ.get("HF_TOKEN")

# Check for Kaggle Secrets if running inside Kaggle Notebook
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HFTOKEN") or user_secrets.get_secret("HF_TOKEN")
    except Exception:
        pass

# Fallback to interactive input if token is not set in environment or secrets
if not hf_token:
    print("[AUTH] HFTOKEN not detected in environment or Kaggle Secrets.")
    hf_token = getpass("Enter your Hugging Face Write Token: ")

os.environ["HFTOKEN"] = hf_token
print(f"[AUTH] Hugging Face Token Configured (Length: {len(hf_token)} chars).")

In [ ]:
# Cell 3: Multi-Phase Dataset Generation Settings (3,500 Shards Total)
DEBUG_MODE = False  # Set to True for a quick test run
REPO_ID = "durgasai299792458/mathmetics-dataset-custom"
SHARD_SIZE = 100000 if not DEBUG_MODE else 1000
OUTPUT_DIR = "/kaggle/working/hf_custom_shards"

import multiprocessing as mp
num_workers = mp.cpu_count()

# 3-Phase Specs:
# Phase 1: 1500 shards, Ints only (int_ratio=1.0), Depths 1-3 (Offset 0 -> 1499)
# Phase 2: 1500 shards, Floats (int_ratio=0.0), Depths 4-6 (Offset 1500 -> 2999)
# Phase 3: 500 shards, Ints only (int_ratio=1.0), Depths 4-6 (Offset 3000 -> 3499)
phases = [
    {"name": "Phase 1: Int-Only Depths 1-3", "shards": 1500 if not DEBUG_MODE else 2, "offset": 0, "min_d": 1, "max_d": 3, "int_ratio": 1.0},
    {"name": "Phase 2: Float Depths 4-6", "shards": 1500 if not DEBUG_MODE else 2, "offset": 1500 if not DEBUG_MODE else 2, "min_d": 4, "max_d": 6, "int_ratio": 0.0},
    {"name": "Phase 3: Int-Only Depths 4-6", "shards": 500 if not DEBUG_MODE else 1, "offset": 3000 if not DEBUG_MODE else 4, "min_d": 4, "max_d": 6, "int_ratio": 1.0}
]

print("=========================================================================")
print(f"[CONFIG] 3-Phase Custom Dataset Generator (3,500 Total Shards)")
print("=========================================================================")
print(f"- Repository: {REPO_ID}")
print(f"- Shard Size: {SHARD_SIZE:,} samples/shard")
print(f"- Phase 1: 1,500 shards (Ints Only, Depths 1-3)")
print(f"- Phase 2: 1,500 shards (Floats Only, Depths 4-6)")
print(f"- Phase 3: 500 shards (Ints Only, Depths 4-6)")
print(f"- CPU Workers: {num_workers}")
print(f"- Debug Mode: {DEBUG_MODE}")
print("=========================================================================")

In [ ]:
# Cell 4: Execute 3-Phase Dataset Generation & Upload
for phase in phases:
    print(f"\n=========================================================================")
    print(f"[EXECUTE] Starting {phase['name']}")
    print(f"- Target Shards: {phase['shards']:,} (Offset: {phase['offset']})")
    print(f"- Depth Range: {phase['min_d']} to {phase['max_d']}")
    print(f"- Int Ratio: {int(phase['int_ratio'] * 100)}%")
    print(f"=========================================================================\n")
    
    cmd = [
        sys.executable, "generate_dataset.py",
        "--num-shards", str(phase["shards"]),
        "--shard-size", str(SHARD_SIZE),
        "--shard-offset", str(phase["offset"]),
        "--min-depth", str(phase["min_d"]),
        "--max-depth", str(phase["max_d"]),
        "--int-ratio", str(phase["int_ratio"]),
        "--repo-id", REPO_ID,
        "--output-dir", OUTPUT_DIR,
        "--num-workers", str(num_workers)
    ]
    if DEBUG_MODE:
        cmd.append("--debug")
        
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f"[ERROR] {phase['name']} failed with return code {proc.returncode}")
        sys.exit(proc.returncode)

print(f"\n=========================================================================")
print(f"[SUCCESS] All 3 Phases Complete! Total 3,500 Shards uploaded to https://huggingface.co/datasets/{REPO_ID}")
print(f"=========================================================================")